# 📊 Customer Segmentation Analysis

**Goal:** Understand customer behaviour and identify distinct customer groups  
**Tools:** Python · SQL (SQLite) · Pandas · Matplotlib · Seaborn  
**Dataset:** 500 Synthetic Customers | Jan 2020 – Dec 2024

---

## Notebook Structure
1. Setup & Data Loading
2. Exploratory Data Analysis (EDA)
3. Customer Segmentation by Age
4. Customer Segmentation by Spending
5. Geographic Analysis
6. RFM Analysis
7. Loyalty & Churn Analysis
8. Visualizations
9. Key Insights & Recommendations

In [ ]:
# ── 1. Setup ──────────────────────────────────────────────────────────────
import sqlite3
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

PALETTE = {
    'Premium': '#1A1A2E', 'Loyal': '#16213E', 'Regular': '#0F3460',
    'New': '#533483', 'Budget': '#E94560', 'At-Risk': '#F5A623'
}
TIER_ORDER = ['Premium', 'Loyal', 'Regular', 'New', 'Budget', 'At-Risk']

sns.set_theme(style='whitegrid', font='DejaVu Sans')
plt.rcParams.update({'figure.dpi': 120, 'savefig.bbox': 'tight',
                     'axes.spines.top': False, 'axes.spines.right': False})
print('✅ Libraries loaded')

In [ ]:
# ── 2. Load Data ───────────────────────────────────────────────────────────
# Run generate_dataset.py first if data/customers.csv doesn't exist
import os
if not os.path.exists('../data/customers.csv'):
    import subprocess
    subprocess.run(['python', '../scripts/generate_dataset.py'], check=True)

df = pd.read_csv('../data/customers.csv', parse_dates=['join_date', 'last_purchase_date'])

# Load into SQLite for SQL queries
conn = sqlite3.connect(':memory:')
df.to_sql('customers', conn, if_exists='replace', index=False)

def sql(query):
    return pd.read_sql_query(query, conn)

print(f'Dataset shape: {df.shape}')
df.head()

In [ ]:
# ── 3. Basic EDA ───────────────────────────────────────────────────────────
print('=== Dataset Overview ===')
print(df.dtypes)
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Numeric Summary ===')
df[['age','total_purchases','total_spent','avg_order_value','loyalty_points']].describe().round(2)

In [ ]:
# Tier distribution
print('=== Customer Tier Distribution ===')
tier_dist = df['customer_tier'].value_counts()
print(tier_dist.to_string())
print(f'\nTotal revenue: ${df["total_spent"].sum():,.2f}')
print(f'Avg CLV:       ${df["total_spent"].mean():,.2f}')

In [ ]:
# ── 4. Age Segmentation (SQL) ──────────────────────────────────────────────
age_seg = sql("""
    SELECT
        CASE
            WHEN age BETWEEN 18 AND 24 THEN '18-24 (Gen Z)'
            WHEN age BETWEEN 25 AND 34 THEN '25-34 (Millennials)'
            WHEN age BETWEEN 35 AND 44 THEN '35-44 (Gen X Young)'
            WHEN age BETWEEN 45 AND 54 THEN '45-54 (Gen X Mature)'
            WHEN age BETWEEN 55 AND 64 THEN '55-64 (Boomers)'
            ELSE '65+ (Seniors)'
        END AS age_group,
        COUNT(*) AS customers,
        ROUND(AVG(total_spent),2) AS avg_spent,
        ROUND(AVG(total_purchases),1) AS avg_purchases,
        ROUND(SUM(total_spent),2) AS group_revenue
    FROM customers
    GROUP BY age_group
    ORDER BY age_group
""")
age_seg

In [ ]:
# ── 5. Spending Band Segmentation ──────────────────────────────────────────
spend_seg = sql("""
    SELECT
        CASE
            WHEN total_spent >= 10000 THEN 'Elite (≥$10K)'
            WHEN total_spent >=  5000 THEN 'Premium ($5K–$10K)'
            WHEN total_spent >=  2000 THEN 'High ($2K–$5K)'
            WHEN total_spent >=   500 THEN 'Mid ($500–$2K)'
            WHEN total_spent >=   100 THEN 'Low ($100–$500)'
            ELSE 'Minimal (<$100)'
        END AS spend_band,
        COUNT(*) AS customers,
        ROUND(100.0*COUNT(*)/500,1) AS pct,
        ROUND(SUM(total_spent),2) AS revenue
    FROM customers
    GROUP BY spend_band
    ORDER BY MIN(total_spent) DESC
""")
spend_seg

In [ ]:
# ── 6. RFM Segmentation ────────────────────────────────────────────────────
rfm = sql("""
    WITH s AS (
        SELECT *,
            CASE WHEN days_since_last_purchase<=30 THEN 5
                 WHEN days_since_last_purchase<=90 THEN 4
                 WHEN days_since_last_purchase<=180 THEN 3
                 WHEN days_since_last_purchase<=365 THEN 2 ELSE 1 END AS R,
            CASE WHEN total_purchases>=100 THEN 5 WHEN total_purchases>=50 THEN 4
                 WHEN total_purchases>=25  THEN 3 WHEN total_purchases>=10 THEN 2 ELSE 1 END AS F,
            CASE WHEN total_spent>=10000 THEN 5 WHEN total_spent>=5000 THEN 4
                 WHEN total_spent>=2000  THEN 3 WHEN total_spent>=500  THEN 2 ELSE 1 END AS M
        FROM customers
    )
    SELECT
        CASE WHEN (R+F+M)>=13 THEN 'Champions'
             WHEN (R+F+M)>=10 THEN 'Loyal Customers'
             WHEN (R+F+M)>=8  THEN 'Potential Loyalists'
             WHEN (R+F+M)>=6  THEN 'At Risk'
             WHEN (R+F+M)>=4  THEN 'Hibernating'
             ELSE 'Lost' END AS rfm_segment,
        COUNT(*) AS count,
        ROUND(AVG(total_spent),2) AS avg_spent
    FROM s
    GROUP BY rfm_segment
    ORDER BY avg_spent DESC
""")
rfm

In [ ]:
# ── 7. At-Risk & Loyalty Analysis ──────────────────────────────────────────
at_risk = sql("""
    SELECT customer_id, full_name, city, ROUND(total_spent,2) AS total_spent,
           days_since_last_purchase, total_purchases, customer_tier,
           CASE WHEN total_spent>=2000 THEN 'HIGH' WHEN total_spent>=500 THEN 'MID' ELSE 'LOW' END AS priority
    FROM customers
    WHERE days_since_last_purchase > 180 AND total_purchases >= 5
    ORDER BY total_spent DESC
    LIMIT 20
""")
print(f'At-Risk customers (inactive >180 days, 5+ purchases): {len(at_risk)}')
at_risk.head(10)

In [ ]:
# ── 8. Visualizations ──────────────────────────────────────────────────────
# Run this cell to generate all charts (also run scripts/run_analysis.py)
import subprocess, sys
result = subprocess.run([sys.executable, '../scripts/run_analysis.py'],
                        capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

In [ ]:
# Display key charts inline
from IPython.display import Image, display
import glob

chart_files = sorted(glob.glob('../outputs/charts/*.png'))
print(f'Found {len(chart_files)} charts:')
for i, path in enumerate(chart_files, 1):
    print(f'  {i}. {os.path.basename(path)}')

# Show first 4 charts
for path in chart_files[:4]:
    print(f'\n### {os.path.basename(path)}')
    display(Image(filename=path, width=700))

## 9. Key Insights & Recommendations

### 🔴 Critical Actions
1. **Win-Back At-Risk High-Value customers** — 105 customers, avg spend > $1,800, inactive >180 days
2. **Protect Premium tier** — 43 customers drive the highest revenue; launch VIP program immediately

### 🟡 Growth Opportunities  
3. **Accelerate New customer conversion** — 86 new customers need a structured 90-day onboarding journey
4. **Upsell Regular → Loyal** — 103 Regular customers are 1–2 targeted campaigns away from Loyal status

### 🟢 Optimisations
5. **Category personalisation** — Tailor communications based on `preferred_category` per segment
6. **Geographic focus** — Concentrate marketing budget on top 10 revenue cities
7. **Loyalty acceleration** — Double points for Budget customers approaching Bronze threshold

See `insights/recommendations.md` for the full analysis.